# Route 97 Road Segment Congestion via Delays

Estimate congestion from **delay gained or recovered while traversing each directed stop-to-stop segment**, rather than bus speed or total lateness. For the same trip instance, `segment delay change = downstream arrival_delay - upstream arrival_delay`.

- **Positive:** the segment added delay.
- **Zero:** no additional delay, even if the bus was already late.
- **Negative:** delay recovery relative to schedule.
- **Missing:** unknown, not zero congestion.

The latest reported arrival-delay estimate is an approved **ground-truth proxy**, not an independently measured arrival. Arrival-to-arrival differences include upstream dwell, holding, and schedule slack, so the result is a delay-accumulation traffic proxy, not proof of pure road congestion.

## Requirements and scope

Run with the `transit_env` Conda environment. Install the existing notebook dependencies from the repository with `cd forecasting && pip install -r requirements.txt` if needed.

This notebook keeps the original analysis scope: bus route **97** (`97-KEL` in GTFS), direction **0**, five-minute UTC bins, and a selected saved backtest. Choose the snapshot explicitly with `backtest_dataset_id` below. Cache mode makes no database or download calls and uses the snapshot's static GTFS. Plotly maps may fetch base-map tiles when opened in a browser.

Reusable selection, geometry, pairing, aggregation, and replay logic lives in `forecasting`; the original speed-based notebook is unchanged.

In [ ]:
from dataclasses import replace
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from forecasting import (
    get_backtest_dataset,
    list_backtest_datasets,
    make_delay_congestion_map,
    prepare_delay_congestion_snapshots,
    run_delay_congestion_engine,
)

In [ ]:
workspace_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
repo_root = next(
    (candidate for candidate in workspace_candidates
     if (candidate / 'forecasting' / 'forecasting').is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate the repository forecasting package.')

backtest_cache_dir = repo_root / 'data' / 'backtests'
visuals_dir = repo_root / 'notebooks' / 'visuals'
visuals_dir.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo_root}')

## Download backtest data

Reuse the original notebook's `get_backtest_dataset` workflow and selected snapshot:

- **`'cache'`** (default): replay the specified dataset offline. A missing/corrupt snapshot raises an error; it never triggers a replacement download. Only load trusted pickle snapshots.
- **`'download'`**: explicitly download a new 48-hour realtime window and current static GTFS, saving an automatically named snapshot. This mode alone reads `.env.road_segment_congestion` or `TRANSIT_DB_*` credentials. Source `./setup_road_segment_congestion_env.sh` from the repository root to configure credentials if needed.

Changing the dataset is an explicit choice. Catalog order is save time, not data coverage. Cached analyses always use the selected snapshot's own static GTFS.

In [ ]:
backtest_source = 'cache'
#backtest_dataset_id = '2026-09-08_to_2026-09-22_legacy'
backtest_dataset_id = "20261026T010808465257Z"
backtest_lookback_hours = 48

display(list_backtest_datasets(backtest_cache_dir))
backtest_dataset = get_backtest_dataset(
    repo_root / 'data',
    source=backtest_source,
    dataset_id=backtest_dataset_id,
    lookback_hours=backtest_lookback_hours,
    env_file=repo_root / '.env.road_segment_congestion',
)
df_vehicle_updates = backtest_dataset.vehicle_updates
df_trip_updates = backtest_dataset.trip_updates
static_data_dir = backtest_dataset.static_data_dir
print(f'Using backtest dataset: {backtest_dataset.directory.name}')
print(f'Vehicle/trip updates: {len(df_vehicle_updates):,} / {len(df_trip_updates):,}')
print(f'Trip-update UTC window: {df_trip_updates.feed_timestamp.min()} through {df_trip_updates.feed_timestamp.max()}')

In [ ]:
id_dtypes = {'stop_id': 'string', 'trip_id': 'string', 'shape_id': 'string', 'route_id': 'string'}
df_stops = pd.read_csv(static_data_dir / 'stops.txt', dtype=id_dtypes)
df_stop_times = pd.read_csv(static_data_dir / 'stop_times.txt', dtype=id_dtypes)
df_trips = pd.read_csv(static_data_dir / 'trips.txt', dtype=id_dtypes)
df_shapes = pd.read_csv(static_data_dir / 'shapes.txt', dtype=id_dtypes)
print(f'Static GTFS snapshot: {static_data_dir}')

## Build the delay-proxy congestion engine

Take the latest publication for each `(trip instance, stop ID, stop sequence)`; greatest database ID breaks timestamp ties, otherwise first input row wins. Missing latest delays remain missing, with no fallback to older predictions.

Trip-start descriptors distinguish daily trips, including crossings of Vancouver midnight. Missing starts are excluded explicitly instead of approximately pairing visits. Each endpoint must match its own trip, direction, stop ID, and sequence in static GTFS. Opposite directions, different shapes, and express stop pairs remain separate. Adjacent scheduled sequences need not differ numerically by one.

Missing visits, missing arrival timestamps, obviously unfinished visits (proxy arrival after the recording window), and nonincreasing endpoint arrival times are audited and excluded. An absent intermediate stop is never bridged to invent a single-segment delta.

In [ ]:
route_id = '97-KEL'
trip_direction_id = 0
time_bin_size = '5min'
max_congestion_age = pd.Timedelta(minutes=15)
congestion_smoothing_window = pd.Timedelta(minutes=50)
animation_window = pd.Timedelta(days=1)
animation_frame_duration_ms = 250
color_limit_seconds = 120

In [ ]:
delay_result = run_delay_congestion_engine(
    df_trip_updates,
    df_stops,
    df_stop_times,
    df_trips,
    df_shapes,
    route_id=route_id,
    trip_direction_id=trip_direction_id,
    time_bin_size=time_bin_size,
)
segments = delay_result.segments
df_traversals = delay_result.traversals
df_valid_traversals = df_traversals.loc[df_traversals.exclusion_reason.eq('included')]
df_congestion = delay_result.congestion
display(delay_result.audit)
print(f'Directed segments: {len(segments)}')
print(f'Usable traversals / candidate traversals: {len(df_valid_traversals):,} / {len(df_traversals):,}')
print(f'Observed segment bins: {len(df_congestion):,}')
print(f'Latest usable proxy crossing: {df_congestion.observed_at.max()}')
print(f'Recording end: {delay_result.observation_end}')

### Inspect coverage before interpreting congestion

Warnings and the audit are part of the analysis. Long realtime windows can contain trips absent from their cached static GTFS; these are not reassigned to a convenient reference trip. Inspect exclusions and the last usable crossing rather than assuming the whole recording period has coverage.

The legacy snapshot has no trip-update receipt timestamps, so historical availability uses publication time with an explicit warning. If future snapshots include receipts, the engine uses the later of publication and receipt. In both cases, a traversal also waits until its downstream proxy arrival.

In [ ]:
segment_coverage = df_valid_traversals.groupby('segment_id').size().rename('traversal_count')
display(segments[
    ['segment_id', 'trip_direction_id', 'shape_id', 'from_stop_name', 'to_stop_name']
].join(segment_coverage, on='segment_id'))
display(delay_result.stop_visits.loc[
    delay_result.stop_visits.exclusion_reason.ne('included'),
    ['trip_id', 'trip_start_timestamp', 'stop_id', 'stop_sequence', 'exclusion_reason'],
].head(10))
display(df_traversals.loc[
    df_traversals.exclusion_reason.ne('included'),
    ['trip_instance_key', 'from_stop_id', 'to_stop_id', 'exclusion_reason'],
].head(10))

## Example trip: accumulated delay versus segment contribution

Choose a trip instance below, or use the one with the most usable segment traversals. This is a retrospective explanation using final reported proxies, not an earlier point-in-time prediction. The upper panel shows total arrival delay; the lower panel shows only the delay gained or recovered on each incoming segment.

In [ ]:
example_trip_instance_key = None
if example_trip_instance_key is None:
    example_trip_instance_key = df_valid_traversals.groupby('trip_instance_key').size().idxmax()
example_visits = delay_result.stop_visits.loc[
    delay_result.stop_visits.trip_instance_key.eq(example_trip_instance_key)
    & delay_result.stop_visits.exclusion_reason.eq('included')
].sort_values('stop_sequence')
example_traversals = df_valid_traversals.loc[
    df_valid_traversals.trip_instance_key.eq(example_trip_instance_key)
].sort_values('to_stop_sequence')
if example_traversals.empty:
    raise ValueError('Selected example trip has no usable segment traversals.')
display(example_traversals[
    ['from_stop_id', 'to_stop_id', 'from_arrival_delay', 'to_arrival_delay',
     'delta_delay_seconds', 'crossed_at', 'available_at']
])

In [ ]:
example_figure = make_subplots(rows=2, cols=1, shared_xaxes=True)
example_figure.add_trace(go.Scatter(
    x=example_visits.stop_sequence, y=example_visits.arrival_delay / 60,
    mode='lines+markers', name='Accumulated arrival delay (proxy)',
), row=1, col=1)
example_figure.add_trace(go.Bar(
    x=example_traversals.to_stop_sequence,
    y=example_traversals.delta_delay_seconds / 60,
    name='Delay change on incoming segment',
), row=2, col=1)
example_figure.add_hline(y=0, line_dash='dash', row=1, col=1)
example_figure.add_hline(y=0, line_dash='dash', row=2, col=1)
example_figure.update_yaxes(title_text='Total delay (min)', row=1, col=1)
example_figure.update_yaxes(title_text='Segment change (min)', row=2, col=1)
example_figure.update_xaxes(title_text='Own-trip stop sequence', row=2, col=1)
example_figure.update_layout(title=f'Arrival-delay proxy decomposition: {example_trip_instance_key}', height=650)
example_figure.write_html(visuals_dir / 'route_97_delay_proxy_trip_example.html')
example_figure.show()

## Segment estimates and historical replay

Each completed traversal contributes **once** to its downstream arrival's five-minute UTC bin. The primary metric is the **median signed delay change in seconds**, not a clipped 0-1 speed index. Counts, spread, and positive/negative/unchanged shares reveal support and disagreement; a zero median can conceal buses experiencing opposite changes. A one-traversal bin is low-support evidence, and its standard deviation is undefined.

A final bin becomes available only after **all contributing endpoint evidence and proxy crossings**. No partial or future final bin is shown early. Replay selects the latest crossed bin already available; late backfilled older bins cannot replace newer crossings. Freshness is measured from the last contributing proxy crossing, not a delayed publication.

Raw and smoothed maps share the same display times and 15-minute provisional expiry. Smoothing is a trailing 50-minute mean of available actual bin medians, not repeated forward-filled frames. It never refreshes timestamps. Equivalent geometries display the freshest shape-specific source without averaging shapes.

The symmetric +/-120-second colour limit controls visual saturation only; full values remain in hover labels. Green means recovery, yellow means zero change, red means added delay, and grey means stale/unknown. These limits and windows are exploratory, not calibrated traffic thresholds.

In [ ]:
display(df_congestion[
    ['segment_id', 'shape_id', 'time_bin', 'median_delta_delay_seconds',
     'traversal_count', 'delta_delay_std_seconds', 'added_delay_share',
     'recovered_delay_share', 'unchanged_delay_share', 'available_at']
].head(10))
display(df_valid_traversals.delta_delay_seconds.describe())

In [ ]:
display_as_of = delay_result.observation_end
display_start = display_as_of - animation_window
display_times = pd.date_range(display_start, display_as_of, freq=time_bin_size)
display_times = display_times.union(pd.DatetimeIndex([display_as_of]))
raw_display = prepare_delay_congestion_snapshots(
    delay_result, display_times, max_age=max_congestion_age,
)
smoothed_display = prepare_delay_congestion_snapshots(
    delay_result, display_times, max_age=max_congestion_age,
    smoothing_window=congestion_smoothing_window,
)
latest_display = raw_display.loc[raw_display.as_of.eq(display_as_of)]
display(latest_display[
    ['segment_id', 'from_stop_name', 'to_stop_name', 'status',
     'last_delta_delay_seconds', 'observed_at', 'age', 'source_shape_id']
])

### Latest map

The map is anchored at the **recording end**, not silently moved to the last good observation. The selected legacy snapshot can therefore end with stale readings when later trips have no matching static GTFS. Change `display_as_of` explicitly above to examine an earlier period; grey segments are not evidence of free flow.

In [ ]:
latest_figure = make_delay_congestion_map(
    latest_display, delay_result.route_geometry,
    title=f'Route {route_id}, direction {trip_direction_id}: latest fresh delay-proxy changes',
    color_limit_seconds=color_limit_seconds,
)
latest_figure.write_html(visuals_dir / 'route_97_segment_congestion_via_delays.html')
latest_figure.show()

In [ ]:
raw_animation = make_delay_congestion_map(
    raw_display, delay_result.route_geometry,
    title=f'Route {route_id}: raw segment delay-proxy changes over {animation_window}',
    color_limit_seconds=color_limit_seconds,
    frame_duration_ms=animation_frame_duration_ms,
)
raw_animation.write_html(visuals_dir / 'route_97_segment_delay_changes_animation.html')
raw_animation.show()

In [ ]:
smoothed_animation = make_delay_congestion_map(
    smoothed_display, delay_result.route_geometry,
    title=f'Route {route_id}: trailing {congestion_smoothing_window} delay-proxy changes',
    color_limit_seconds=color_limit_seconds,
    frame_duration_ms=animation_frame_duration_ms,
)
smoothed_animation.write_html(visuals_dir / 'route_97_segment_delay_changes_smoothed_animation.html')
smoothed_animation.show()

## One shape-specific segment: signal and traversal support

Choose an engine segment ID from the coverage table. The default has the most usable traversals. The upper panel is availability-aware replay for this exact shape, with stale/unknown gaps left blank. The lower panel is a **retrospective coverage diagnostic**: final distinct traversals per crossing bin, not repeated carried-forward frame counts and not an earlier-known feature. Hovering a count shows when its final evidence became available. Timestamps are displayed in Vancouver time.

In [ ]:
segment_id = int(segment_coverage.idxmax())
selected_segments = segments.loc[segments.segment_id.eq(segment_id)]
if selected_segments.empty:
    raise ValueError('Selected segment ID is not in this engine result.')
segment_bins = df_congestion.loc[df_congestion.segment_id.eq(segment_id)]
selected_result = replace(delay_result, segments=selected_segments, congestion=segment_bins)
segment_raw = prepare_delay_congestion_snapshots(
    selected_result, display_times, max_age=max_congestion_age,
)
segment_smooth = prepare_delay_congestion_snapshots(
    selected_result, display_times, max_age=max_congestion_age,
    smoothing_window=congestion_smoothing_window,
)
segment_counts = segment_bins.loc[segment_bins.time_bin.between(display_start.floor(time_bin_size), display_as_of)]
selected_segment = selected_segments.iloc[0]
segment_label = f'{selected_segment.from_stop_name} -> {selected_segment.to_stop_name} | shape {selected_segment.shape_id}'

In [ ]:
segment_figure = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3])
segment_figure.add_trace(go.Scatter(
    x=segment_raw.as_of.dt.tz_convert('America/Vancouver'),
    y=segment_raw.delta_delay_seconds / 60,
    mode='lines+markers', name='Raw median', connectgaps=False,
), row=1, col=1)
segment_figure.add_trace(go.Scatter(
    x=segment_smooth.as_of.dt.tz_convert('America/Vancouver'),
    y=segment_smooth.delta_delay_seconds / 60,
    mode='lines+markers', name='Trailing 50-minute mean', connectgaps=False,
), row=1, col=1)
segment_figure.add_trace(go.Bar(
    x=segment_counts.time_bin.dt.tz_convert('America/Vancouver'),
    y=segment_counts.traversal_count, name='Final traversals per crossing bin',
    customdata=segment_counts.available_at.astype('string'),
    hovertemplate='%{x}<br>Traversals: %{y}<br>Final evidence available (UTC): %{customdata}<extra></extra>',
), row=2, col=1)
segment_figure.add_hline(y=0, line_dash='dash', row=1, col=1)
segment_figure.update_yaxes(title_text='Delay change (min)', row=1, col=1)
segment_figure.update_yaxes(title_text='Distinct traversals', row=2, col=1)
segment_figure.update_xaxes(title_text='Time (Vancouver)', row=2, col=1)
segment_figure.update_layout(title=f'Delay-proxy congestion and coverage: {segment_label}', height=700)
segment_figure.write_html(visuals_dir / 'route_97_segment_delay_changes_timeseries.html')
segment_figure.show()

## Sanity checks and limitations

Check exact endpoint subtraction, unique scheduled traversals, direction isolation, missing exclusions, and no evidence from after a replay timestamp. Package tests also exercise inherited lateness, early buses, duplicate updates, missing stops, midnight trips, shape variants, freshness boundaries, and smoothing.

Positive changes can arise from traffic, dwell, disruptions, or prediction revisions. Negative changes indicate recovery relative to the timetable, not guaranteed absolute free flow. Zero means no added delay relative to schedule, not necessarily an empty road. Schedule slack and holding remain confounders. Final reported arrival timestamps also determine bin placement and can still be imperfect, even after excluding obviously unfinished visits.

In [ ]:
assert df_valid_traversals.delta_delay_seconds.eq(
    df_valid_traversals.to_arrival_delay - df_valid_traversals.from_arrival_delay
).all()
assert not df_valid_traversals.duplicated(['trip_instance_key', 'to_stop_sequence']).any()
assert df_valid_traversals.trip_direction_id.eq(trip_direction_id).all()
assert df_traversals.loc[df_traversals.exclusion_reason.ne('included'), 'delta_delay_seconds'].isna().all()
known = raw_display.loc[raw_display.observed_at.notna()]
assert known.available_at.le(known.as_of).all()
assert raw_display.loc[raw_display.status.ne('fresh'), 'delta_delay_seconds'].isna().all()
print('Delay subtraction, coverage, direction, and historical availability checks passed.')